In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/tabular-playground-series-apr-2022/sample_submission.csv
/kaggle/input/competitions/tabular-playground-series-apr-2022/train_labels.csv
/kaggle/input/competitions/tabular-playground-series-apr-2022/train.csv
/kaggle/input/competitions/tabular-playground-series-apr-2022/test.csv


In [2]:
train = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/train.csv')
test = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/test.csv')
labels = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/train_labels.csv')
sub = pd.read_csv('/kaggle/input/competitions/tabular-playground-series-apr-2022/sample_submission.csv')

In [3]:
train.describe()

,sequence,subject,step,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12
count,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06,1.558080e+06
mean,1.298350e+04,3.316331e+02,2.950000e+01,4.365526e-04,-1.034982e-03,-2.178045e-01,-2.156555e-03,-1.828903e-03,-1.651785e-03,-4.122917e-04,-2.620665e-05,-1.298393e-04,1.365584e-03,3.315801e-04,-3.733291e-03,-1.172605e-02
std,7.496318e+03,1.958257e+02,1.731811e+01,2.658684e+00,4.404200e+00,2.298002e+00,3.934184e+00,1.683685e+00,1.590818e+00,3.345143e+00,3.243428e+00,4.501534e+00,2.592913e+00,1.917333e+00,4.532568e+00,3.911767e+01
min,0.000000e+00,0.000000e+00,0.000000e+00,-3.750634e+02,-4.345977e+02,-3.165948e+01,-4.083761e+02,-2.362601e+01,-7.498280e+01,-4.705046e+02,-4.070115e+02,-5.361000e+02,-2.703468e+02,-4.341271e+01,-4.270586e+02,-6.125494e+02
25%,6.491750e+03,1.617500e+02,1.475000e+01,-5.000000e-01,-4.831933e-01,-6.461531e-01,-4.929204e-01,-4.729928e-01,-4.786836e-01,-4.927140e-01,-5.022901e-01,-5.000000e-01,-5.151734e-01,-4.787939e-01,-4.835391e-01,-5.805627e-01
50%,1.298350e+04,3.350000e+02,2.950000e+01,-3.091190e-03,3.151261e-03,0.000000e+00,0.000000e+00,-1.589577e-03,2.991773e-03,9.107468e-04,-2.290076e-03,0.000000e+00,-1.445087e-03,-1.655822e-03,3.086420e-03,0.000000e+00
75%,1.947525e+04,5.010000e+02,4.425000e+01,4.845440e-01,4.926471e-01,3.338469e-01,4.893805e-01,4.701565e-01,5.056096e-01,4.927140e-01,4.847328e-01,5.000000e-01,5.086705e-01,4.780386e-01,4.938272e-01,5.703325e-01
max,2.596700e+04,6.710000e+02,5.900000e+01,3.358246e+02,4.495914e+02,1.666667e+00,4.366504e+02,2.487286e+01,7.791548e+01,4.425009e+02,3.312542e+02,6.301000e+02,3.679812e+02,4.186559e+01,4.480206e+02,6.305111e+02


In [4]:
train.head()

,sequence,subject,step,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12
0,0,47,0,-0.196291,0.112395,1.0,0.329204,-1.004660,-0.131638,-0.127505,0.368702,-0.1,-0.963873,-0.985069,0.531893,4.751492
1,0,47,1,-0.447450,0.134454,1.0,-0.658407,0.162495,0.340314,-0.209472,-0.867176,0.2,-0.301301,0.082733,-0.231481,0.454390
2,0,47,2,0.326893,-0.694328,1.0,0.330088,0.473678,1.280479,-0.094718,0.535878,1.4,1.002168,0.449221,-0.586420,-4.736147
3,0,47,3,0.523184,0.751050,1.0,0.976991,-0.563287,-0.720269,0.793260,0.951145,-0.3,-0.995665,-0.434290,1.344650,0.429241
4,0,47,4,0.272025,1.074580,1.0,-0.136283,0.398579,0.044877,0.560109,-0.541985,-0.9,1.055636,0.812631,0.123457,-0.223359


In [5]:
print(train.groupby('sequence').size().value_counts())   # 시퀀스 길이가 모두 60?
print(train['subject'].nunique(), test['subject'].nunique())
print(labels['state'].value_counts(normalize=True))       # 클래스 비율

60    25968
Name: count, dtype: int64
672 319
state
1    0.501155
0    0.498845
Name: proportion, dtype: float64


In [6]:
sensors = [c for c in train.columns if c.startswith('sensor')]

def make_features(df):
    g = df.groupby('sequence')[sensors]
    feat = g.agg(['mean', 'std', 'min', 'max', 'median', 'skew'])
    diff = df[sensors].diff().where(df['step']!=0)
    
    diff_feat = diff.groupby(df['sequence']).agg(['mean','std','max'])
    feat = feat.join(diff_feat, rsuffix='_diff')
    feat_columns = ['_'.join(c) for c in feat.columns]
    return feat

중요 : 사람(subject)이 test와 train에서 겹치지 않아서 train 시 evaluation 때도 subject를 구분해야.

In [7]:
X = make_features(train)
y = labels.set_index('sequence').loc[X.index, 'state']
groups = train.groupby('sequence')['subject'].first().loc[X.index]
X_test = make_features(test).loc[sub['sequence']]              # 제출 파일 순서에 맞춤

X.columns = ['_'.join(map(str, c)) if isinstance(c, tuple) else str(c) for c in X.columns]
X.columns = X.columns.str.replace(r'[^\w]', '_', regex=True)
X_test.columns = X.columns

In [8]:
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(X))
test_pred = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(gkf.split(X, y, groups)):
    model = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.03, verbose=-1)
    model.fit(X.iloc[tr], y.iloc[tr], 
              eval_set=[(X.iloc[va], y.iloc[va])], eval_metric='auc',
              callbacks=[lgb.early_stopping(100, verbose=False)])

    oof[va] = model.predict_proba(X.iloc[va])[:, 1]
    test_pred += model.predict_proba(X_test)[:, 1] / gkf.n_splits
    print(f'Fold {fold} AUC: {roc_auc_score(y.iloc[va], oof[va]):.4f}')

print(f'CV AUC: {roc_auc_score(y, oof):.4f}')

Fold 0 AUC: 0.9137
Fold 1 AUC: 0.9135
Fold 2 AUC: 0.9273
Fold 3 AUC: 0.9236
Fold 4 AUC: 0.9246
CV AUC: 0.9204


In [9]:
sub['state'] = test_pred
sub.to_csv('submission.csv', index=False)
print(sub.head())

   sequence     state
0     25968  0.830616
1     25969  0.988480
2     25970  0.012200
3     25971  0.221444
4     25972  0.717299
